# LLM Training Lab, lltGPT step 8: toggle each modern component (T4, fp32)

FROM SCRATCH. A 2.2M-parameter GPT on TinyStories, 600 steps per run. Modern config (RMSNorm, SwiGLU, RoPE, pre-norm) and one swap back to GPT-2's choice per variant, seeds 0 and 1, plus two deliberately broken runs. The code is the lab page's own cells, moved to the GPU, float32.

In [ ]:

import os, platform, shutil, subprocess, sys, time
START = time.time()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers==0.22.1"], check=False)
import torch
print("Python", platform.python_version(), "| torch", torch.__version__)
if not os.environ.get("LLT_SMOKE"):
    assert torch.cuda.is_available(), "No GPU: Settings > Accelerator > GPU T4 x2"
for i in range(torch.cuda.device_count()):
    print("GPU", i, torch.cuda.get_device_name(i), "capability", torch.cuda.get_device_capability(i))
print("Free disk", round(shutil.disk_usage(".").free / 1e9, 1), "GB")


In [ ]:
LIB = 'import torch\nDEV = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n\nimport math\nfrom dataclasses import dataclass, replace\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n@dataclass\nclass Config:\n    vocab: int = 50257\n    ctx: int = 1024\n    d: int = 768\n    layers: int = 12\n    heads: int = 12\n    norm: str = "rmsnorm"   # "rmsnorm" or "layernorm"\n    mlp: str = "swiglu"     # "swiglu" or "gelu"\n    pos: str = "rope"       # "rope" or "learned"\n    prenorm: bool = True\n    bias: bool = False\n    eps: float = 1e-5\n\nGPT2 = Config(norm="layernorm", mlp="gelu", pos="learned", bias=True)\nMODERN = Config()\n\nclass RMSNorm(nn.Module):\n    def __init__(self, d, eps):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(d))\n        self.eps = eps\n\n    def forward(self, x):\n        return self.weight * x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)\n\ndef make_norm(cfg):\n    if cfg.norm == "rmsnorm":\n        return RMSNorm(cfg.d, cfg.eps)\n    return nn.LayerNorm(cfg.d, eps=cfg.eps, bias=cfg.bias)\n\ntorch.manual_seed(0)\nx = torch.randn(2, 5, 768) * 3 + 1   # mean 1, std 3\nfor name, norm in [("LayerNorm", make_norm(GPT2)), ("RMSNorm", make_norm(MODERN))]:\n    y = norm(x)\n    n = sum(p.numel() for p in norm.parameters())\n    print(f"{name:9}  mean {y.mean(-1).mean():.4f}  rms {y.pow(2).mean(-1).sqrt().mean():.4f}  parameters {n}")\n\n\ndef rope_tables(T, hd, offset=0, base=10000.0, device=None):\n    inv = base ** (-torch.arange(0, hd, 2, device=device).float() / hd)\n    ang = torch.outer(torch.arange(offset, offset + T, device=device).float(), inv)\n    return ang.cos(), ang.sin()\n\ndef apply_rope(x, cos, sin):\n    # x is (batch, heads, T, head_dim). Rotate the pairs (x[2i], x[2i+1]) as in step 1.\n    x1, x2 = x[..., 0::2], x[..., 1::2]\n    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)\n\nclass Attention(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.heads, self.rope = cfg.heads, cfg.pos == "rope"\n        self.qkv = nn.Linear(cfg.d, 3 * cfg.d, bias=cfg.bias)\n        self.proj = nn.Linear(cfg.d, cfg.d, bias=cfg.bias)\n\n    def forward(self, x, cos=None, sin=None):\n        B, T, C = x.shape\n        q, k, v = self.qkv(x).split(C, dim=2)\n        q, k, v = (t.view(B, T, self.heads, C // self.heads).transpose(1, 2) for t in (q, k, v))\n        if self.rope:\n            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)   # queries and keys, never values\n        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n        return self.proj(y.transpose(1, 2).reshape(B, T, C))\n\ndef swiglu_hidden(d):\n    # 8d/3 keeps the parameter count of a 4d GELU MLP; round up to a multiple of 64\n    return 64 * math.ceil(8 * d / 3 / 64)\n\nclass MLP(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.swiglu = cfg.mlp == "swiglu"\n        hidden = swiglu_hidden(cfg.d) if self.swiglu else 4 * cfg.d\n        self.gate = nn.Linear(cfg.d, hidden, bias=cfg.bias) if self.swiglu else None\n        self.up = nn.Linear(cfg.d, hidden, bias=cfg.bias)\n        self.down = nn.Linear(hidden, cfg.d, bias=cfg.bias)\n\n    def forward(self, x):\n        if self.swiglu:\n            return self.down(F.silu(self.gate(x)) * self.up(x))\n        return self.down(F.gelu(self.up(x), approximate="tanh"))\n\nclass Block(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.prenorm = cfg.prenorm\n        self.n1, self.n2 = make_norm(cfg), make_norm(cfg)\n        self.attn, self.mlp = Attention(cfg), MLP(cfg)\n\n    def forward(self, x, cos=None, sin=None):\n        if self.prenorm:\n            x = x + self.attn(self.n1(x), cos, sin)\n            return x + self.mlp(self.n2(x))\n        x = self.n1(x + self.attn(x, cos, sin))\n        return self.n2(x + self.mlp(x))\n\nfor d in [192, 320, 768, 4096]:\n    h = swiglu_hidden(d)\n    print(f"d = {d:>4}  8d/3 = {8 * d / 3:7.1f}  hidden {h:>5}  MLP weights {3 * d * h:>11,}  GELU 4d MLP {8 * d * d:>11,}")\n\n\nclass GPT(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.tok = nn.Embedding(cfg.vocab, cfg.d)\n        self.pos = nn.Embedding(cfg.ctx, cfg.d) if cfg.pos == "learned" else None\n        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.layers)])\n        self.norm = make_norm(cfg)\n        self.head = nn.Linear(cfg.d, cfg.vocab, bias=False)\n        self.head.weight = self.tok.weight   # tie first, so the init below sets the shared tensor once\n        for name, p in self.named_parameters():\n            if name.endswith(("proj.weight", "down.weight")):\n                # the two layers per block that write into the residual stream\n                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * cfg.layers))\n            elif p.dim() == 2:\n                nn.init.normal_(p, 0.0, 0.02)\n            elif name.endswith("bias"):\n                nn.init.zeros_(p)\n\n    def forward(self, idx, offset=0):\n        B, T = idx.shape\n        x = self.tok(idx)\n        cos = sin = None\n        if self.pos is None:\n            cos, sin = rope_tables(T, self.cfg.d // self.cfg.heads, offset, device=idx.device)\n        else:\n            x = x + self.pos(torch.arange(offset, offset + T, device=idx.device))\n        for block in self.blocks:\n            x = block(x, cos, sin)\n        return self.head(self.norm(x))\n\n# The model our TinyStories pretraining run uses (lltTinyStories)\nTINY = replace(MODERN, vocab=4096, ctx=512, d=320, layers=8, heads=8)\ntorch.manual_seed(0)\nmodel = GPT(TINY)\nprint("head is the embedding tensor:", model.head.weight is model.tok.weight)\nprint(f"init std  qkv {model.blocks[0].attn.qkv.weight.std():.4f}  proj {model.blocks[0].attn.proj.weight.std():.4f}"\n      f"  down {model.blocks[0].mlp.down.weight.std():.4f}  target 0.02/sqrt(16) = {0.02 / math.sqrt(16):.4f}")\nidx, targets = torch.randint(0, TINY.vocab, (2, 4, 128))\nwith torch.no_grad():\n    loss = F.cross_entropy(model(idx).flatten(0, 1), targets.flatten())\nprint(f"step-0 loss {loss:.3f}  ln(V) = {math.log(TINY.vocab):.3f}")\n\n\ndef numel(model):\n    return sum(p.numel() for p in model.parameters())   # a tied tensor counts once\n\ndef approx_params(cfg):\n    return 12 * cfg.layers * cfg.d ** 2 + cfg.vocab * cfg.d\n\nTOGGLE = replace(MODERN, vocab=2048, ctx=128, d=192, layers=4, heads=6)   # the step 8 experiment\nfor name, cfg in [("GPT-2 124M", GPT2), ("modern, GPT-2 size", MODERN), ("TinyStories model", TINY), ("step 8 model", TOGGLE),\n                  ("step 8, 50k vocab", replace(TOGGLE, vocab=50257))]:\n    n, a, emb = numel(GPT(cfg)), approx_params(cfg), cfg.vocab * cfg.d\n    print(f"{name:18}  sum(numel) {n:>11,}  12Ld^2 + Vd {a:>11,}  gap {(n - a) / n:+.2%}  embedding {emb / n:.0%}")\n\n\nfrom huggingface_hub import hf_hub_download\nfrom tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers\n\npath = hf_hub_download("roneneldan/TinyStories", "TinyStoriesV2-GPT4-valid.txt", repo_type="dataset")\nstories = [s.strip() for s in open(path, encoding="utf-8").read().split("<|endoftext|>") if s.strip()]\nsplit = int(len(stories) * 0.95)   # 95% to train on, the last 5% held out\nbpe = Tokenizer(models.BPE())\nbpe.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)\nbpe.decoder = decoders.ByteLevel()\nbpe.train_from_iterator(stories[:split], trainers.BpeTrainer(vocab_size=TOGGLE.vocab, special_tokens=["<|eot|>"],\n                                                             initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))\n\ndef encode(texts):\n    ids = []\n    for enc in bpe.encode_batch(texts):\n        ids += enc.ids + [bpe.token_to_id("<|eot|>")]\n    return torch.tensor(ids)\n\ntrain_ids, val_ids = encode(stories[:split]), encode(stories[split:])\nVAL_X = val_ids[: 256 * 128].view(256, 128)          # 256 fixed windows of 128 tokens\nVAL_Y = val_ids[1 : 256 * 128 + 1].view(256, 128)\nVAL_X, VAL_Y = VAL_X.to(DEV), VAL_Y.to(DEV)\nprint(f"{len(stories):,} stories  train tokens {len(train_ids):,}  val tokens {len(val_ids):,}")\n\n@torch.no_grad()\ndef val_loss(model):\n    model.eval()\n    loss = sum(F.cross_entropy(model(x).flatten(0, 1), y.flatten()).item() for x, y in zip(VAL_X.split(64), VAL_Y.split(64))) / 4\n    model.train()\n    return loss\n\ndef train(cfg, steps=600, seed=0, lr=3e-3, batch=32, warmup=50, every=100):\n    torch.manual_seed(seed)\n    model, gen = GPT(cfg).to(DEV), torch.Generator().manual_seed(seed)\n    groups = [{"params": [p for p in model.parameters() if p.dim() >= 2], "weight_decay": 0.1},\n              {"params": [p for p in model.parameters() if p.dim() < 2], "weight_decay": 0.0}]\n    opt = torch.optim.AdamW(groups, lr=lr, betas=(0.9, 0.95))\n    curve = [(0, val_loss(model))]\n    for step in range(steps):\n        frac = (step + 1) / warmup if step < warmup else 0.1 + 0.45 * (1 + math.cos(math.pi * (step - warmup) / (steps - warmup)))\n        for g in opt.param_groups:\n            g["lr"] = lr * frac\n        starts = torch.randint(0, len(train_ids) - cfg.ctx - 1, (batch,), generator=gen)\n        x = torch.stack([train_ids[s : s + cfg.ctx] for s in starts])\n        y = torch.stack([train_ids[s + 1 : s + cfg.ctx + 1] for s in starts])\n        x, y = x.to(DEV), y.to(DEV)\n        loss = F.cross_entropy(model(x).flatten(0, 1), y.flatten())\n        opt.zero_grad(set_to_none=True)\n        loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)\n        opt.step()\n        if (step + 1) % every == 0:\n            curve.append((step + 1, val_loss(model)))\n    return model, curve\n'
import os
WORK = os.environ.get('LLT_WORKDIR', '/kaggle/working')
open(os.path.join(WORK, 'lltgpt.py'), 'w').write(LIB)
WORKER = '\nimport json, math, os, platform, sys, time\nWORK = os.environ.get("LLT_WORKDIR", "/kaggle/working")\nos.chdir(WORK)\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nns = {"__name__": "lltgpt"}\nt0 = time.time()\nexec(compile(open("lltgpt.py", encoding="utf-8").read(), "lltgpt.py", "exec"), ns)\ndata_sec = time.time() - t0\nGPT, train, numel, replace, TOGGLE = ns["GPT"], ns["train"], ns["numel"], ns["replace"], ns["TOGGLE"]\nDEV = ns["DEV"]\nCUDA = torch.cuda.is_available()\nSTEPS = int(os.environ.get("LLT_STEPS", "600"))\njobs = json.load(open(sys.argv[1]))\n\ndef attn_rope_v(self, x, cos=None, sin=None):\n    B, T, C = x.shape\n    q, k, v = self.qkv(x).split(C, dim=2)\n    q, k, v = (t.view(B, T, self.heads, C // self.heads).transpose(1, 2) for t in (q, k, v))\n    q, k, v = (ns["apply_rope"](t, cos, sin) for t in (q, k, v))\n    y = F.scaled_dot_product_attention(q, k, v, is_causal=True)\n    return self.proj(y.transpose(1, 2).reshape(B, T, C))\n\ndef gpt_rope_v(cfg):\n    m = GPT(cfg)\n    for b in m.blocks:\n        b.attn.forward = attn_rope_v.__get__(b.attn)\n    return m\n\ndef gpt_tie_after(cfg):\n    m = GPT.__new__(GPT)\n    nn.Module.__init__(m)\n    m.cfg = cfg\n    m.tok = nn.Embedding(cfg.vocab, cfg.d)\n    m.pos = nn.Embedding(cfg.ctx, cfg.d) if cfg.pos == "learned" else None\n    m.blocks = nn.ModuleList([ns["Block"](cfg) for _ in range(cfg.layers)])\n    m.norm = ns["make_norm"](cfg)\n    m.head = nn.Linear(cfg.d, cfg.vocab, bias=False)\n    for name, mod in m.named_modules():\n        if isinstance(mod, nn.Linear):\n            std = 0.02 / math.sqrt(2 * cfg.layers) if name.endswith(("proj", "down")) else 0.02\n            nn.init.normal_(mod.weight, 0.0, std)\n            if mod.bias is not None:\n                nn.init.zeros_(mod.bias)\n    m.head.weight = m.tok.weight\n    return m\n\nBUILDERS = {"none": GPT, "rope-on-v": gpt_rope_v, "tie-after-init": gpt_tie_after}\n\ndef shift_test(builder):\n    torch.manual_seed(0)\n    m = builder(replace(ns["MODERN"], vocab=1000, ctx=64, d=64, layers=2, heads=4)).eval()\n    idx = torch.randint(0, 1000, (2, 16))\n    with torch.no_grad():\n        return (m(idx) - m(idx, offset=40)).abs().max().item(), (m(idx) - m(idx)).abs().max().item()\n\nfor job in jobs:\n    cfg = replace(TOGGLE, **job["overrides"])\n    builder = BUILDERS[job["builder"]]\n    extra = {}\n    if job["builder"] == "rope-on-v":\n        extra["shift40MaxLogitChange"], extra["samePassTwiceMaxChange"] = shift_test(gpt_rope_v)\n        extra["shift40Correct"], _ = shift_test(GPT)\n    torch.manual_seed(job["seed"])\n    probe = builder(cfg)\n    extra["headWeightStdAtInit"] = round(probe.head.weight.std().item(), 4)\n    ns["GPT"] = builder\n    if CUDA:\n        torch.cuda.reset_peak_memory_stats()\n    t = time.time()\n    try:\n        model, curve = train(cfg, steps=STEPS, seed=job["seed"])\n        status = "ok"\n    except Exception as e:\n        model, curve, status = None, [], "error: " + repr(e)\n    if CUDA:\n        torch.cuda.synchronize()\n    wall = time.time() - t\n    ns["GPT"] = GPT\n    fin = [v for _, v in curve if v == v]\n    rec = {\n        "runId": job["runId"], "module": "lltGPT", "tier": "T2", "label": job["label"],\n        "hardware": {"gpu": torch.cuda.get_device_name(0) if CUDA else None, "count": 1 if CUDA else 0,\n                     "vramGB": round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1) if CUDA else None,\n                     "capability": list(torch.cuda.get_device_capability()) if CUDA else None, "note": "Kaggle 2x T4 session, one run per GPU"},\n        "env": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,\n                "tokenizers": __import__("tokenizers").__version__, "dtype": "float32, no autocast"},\n        "notebookCommit": None,\n        "config": {**vars(cfg), "steps": STEPS, "batch": 32, "ctx": 128, "lr": 3e-3, "warmup": 50,\n                   "schedule": "linear warmup, cosine to 10%", "optimizer": "AdamW betas (0.9, 0.95), wd 0.1 on 2D params", "clip": 1.0,\n                   "params": numel(model) if model is not None else None, "builder": job["builder"],\n                   "data": {"dataset": "roneneldan/TinyStories TinyStoriesV2-GPT4-valid.txt", "trainTokens": len(ns["train_ids"]),\n                            "valTokens": len(ns["val_ids"]), "valEvalTokens": 256 * 128, "dataPrepSec": round(data_sec, 1)}},\n        "seed": job["seed"], "status": status,\n        "metrics": [{"step": s, "tokens": s * 32 * 128, "valLoss": round(v, 4)} for s, v in curve],\n        "evals": {"finalValLoss": round(curve[-1][1], 4) if curve else None, "bestValLoss": round(min(fin), 4) if fin else None, **extra},\n        "wallClockSec": round(wall, 1), "peakMemGB": round(torch.cuda.max_memory_allocated() / 1e9, 3) if CUDA else None,\n        "costUSD": 0, "priceCheckedOn": None, "date": time.strftime("%Y-%m-%d"),\n    }\n    os.makedirs("runs", exist_ok=True)\n    with open(os.path.join("runs", job["runId"] + ".json"), "w") as f:\n        json.dump(rec, f, indent=1)\n    print(job["runId"], status, "val", [round(v, 3) for _, v in curve], "wall", round(wall, 1), flush=True)\n'
open(os.path.join(WORK, 'worker.py'), 'w').write(WORKER)
print('wrote lltgpt.py', len(LIB), 'chars and worker.py')

In [ ]:

import json
os.chdir(WORK)
V = {"modern": {}, "learned-pos": {"pos": "learned"}, "gelu": {"mlp": "gelu"}, "layernorm": {"norm": "layernorm"},
     "postnorm": {"prenorm": False}, "gpt2": {"norm": "layernorm", "mlp": "gelu", "pos": "learned", "bias": True}}
jobs = [{"runId": f"lltGPT-t2-toggle-{k}-s{s}", "label": "baseline" if k == "modern" else f"ablation:{k}", "overrides": o, "seed": s, "builder": "none"}
        for s in (0, 1) for k, o in V.items()]
jobs += [{"runId": f"lltGPT-t2-broken-{b}", "label": f"broken:{b}", "overrides": {}, "seed": 0, "builder": b} for b in ("rope-on-v", "tie-after-init")]
n = max(1, torch.cuda.device_count())
if os.environ.get("LLT_SMOKE"):
    jobs = [jobs[0], jobs[-2], jobs[-1]]
procs = []
for g in range(n):
    path = f"jobs_{g}.json"
    json.dump(jobs[g::n], open(path, "w"))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    procs.append(subprocess.Popen([sys.executable, "worker.py", path], env=env, stdout=open(f"worker_{g}.log", "w"), stderr=subprocess.STDOUT))
for p in procs:
    p.wait()
for g in range(n):
    print(open(f"worker_{g}.log").read())
print("runs", sorted(os.listdir("runs")), "elapsed min", round((time.time() - START) / 60, 1))
